In [1]:
!pip uninstall -y transformers sentence-transformers faiss-cpu gradio

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: sentence-transformers 5.4.0
Uninstalling sentence-transformers-5.4.0:
  Successfully uninstalled sentence-transformers-5.4.0
Found existing installation: gradio 5.50.0
Uninstalling gradio-5.50.0:
  Successfully uninstalled gradio-5.50.0


In [2]:
!pip install -q \
transformers==4.41.2 \
sentence-transformers==2.7.0 \
faiss-cpu \
gradio==4.36.1 \
pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.1/318.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires web

In [3]:
import transformers, gradio, sentence_transformers

print("Transformers:", transformers.__version__)
print("Gradio:", gradio.__version__)

Transformers: 4.41.2
Gradio: 4.36.1


In [4]:
from google.colab import files
uploaded = files.upload()

Saving mental_health_dataset.csv to mental_health_dataset.csv


In [5]:
import pandas as pd

df = pd.read_csv("mental_health_dataset.csv")
df.columns = df.columns.str.strip()

df = df[['Prompt', 'Solution', 'Severity', 'Intent', 'Keywords']]
df = df.dropna(subset=['Prompt', 'Solution']).reset_index(drop=True)

print("✅ Loaded:", len(df))
df.head()

✅ Loaded: 2005


,Prompt,Solution,Severity,Intent,Keywords
0,What are the early signs of depression?,This is an informational query. It is best to ...,low,query_symptoms,"what, are, the, early"
1,I feel really overwhelmed with my semester exams.,It sounds like you're going through something ...,medium,stress_academic,"i, feel, really, overwhelmed"
2,I hate how my professor treats me in class!,That sounds really tough. Your feelings are va...,medium,vent_anger,"i, hate, how, my"
3,I don't see the point in waking up tomorrow.,I'm really sorry you're feeling this way. You ...,high,crisis_suicidal,"i, don't, see, the"
4,My anxiety is getting worse before presentations.,It sounds like you're going through something ...,medium,anxiety_academic,"my, anxiety, is, getting"


In [6]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embed_model.encode(df['Prompt'].tolist(), show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [7]:
import faiss
import numpy as np

dim = embeddings.shape[1]

index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings))

print("✅ FAISS Ready:", index.ntotal)

✅ FAISS Ready: 2005


In [19]:
def retrieve_context(user_input, top_k=3):
    emb = embed_model.encode([user_input])

    D, I = index.search(np.array(emb), top_k)

    contexts = []
    for i in I[0]:
        contexts.append(f"- {df.iloc[i]['Solution']}")

    return "\n".join(contexts)

In [35]:
import re

# 🔴 Crisis keywords (strong signals only)
crisis_keywords = [
    "suicide","sucide", "suicidal", "sucidal",
    "kill myself", "want to die", "end my life",
    "take my life", "end it all",
    "hurt myself", "self harm", "self-harm",
    "cut myself", "cutting",
    "i don't want to live", "i dont want to live",
    "life is not worth living",
    "better off dead",
    "no reason to live",
    "give up on life",
    "i feel like dying"
]

# 🔴 Regex patterns (for flexible detection)
crisis_patterns = [
    r"kill (myself|me)",
    r"end (my life|it all)",
    r"want to die",
    r"hurt myself",
    r"take my life"
]

# 🚨 Safety check function
def safety_check(text):
    text = text.lower().strip()

    # Keyword match
    if any(keyword in text for keyword in crisis_keywords):
        return True

    # Pattern match
    for pattern in crisis_patterns:
        if re.search(pattern, text):
            return True

    return False


# 🚨 Crisis response (safe + human tone)
def crisis_response():
    return (
        "🚨 I'm really sorry you're feeling this way.\n\n"
        "You’re not alone, and there are people who genuinely care about you.\n\n"
        "📞 Kiran Mental Health Helpline (India): 9152987821\n"
        "📞 Emergency: 112\n\n"
        "If possible, please reach out to a trusted friend, family member, or a professional right now.\n"
        "I'm here to listen — do you want to share what's been going on?"
    )

In [29]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_length=120   # increase to 120 for better response
)

In [31]:
def generate_rag_response(user_input, context):

    prompt = f"""
You are a helpful and empathetic mental health assistant.

A user is sharing their feelings.

User message:
{user_input}

Relevant advice:
{context}

Write a clear, supportive, and complete response in 2-4 sentences.
Do NOT repeat keywords. Do NOT give one-word answers.
Be human-like and caring.
"""

    result = generator(prompt)
    return result[0]['generated_text']

In [32]:
def chatbot(user_input):

    if safety_check(user_input):
        return crisis_response()

    context = retrieve_context(user_input)

    return generate_rag_response(user_input, context)

In [34]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## 🧠 Pandora : Mental Health Chatbot (RAG)")

    inp = gr.Textbox(label="Your Message")
    out = gr.Textbox(label="Response")

    btn = gr.Button("Send")

    btn.click(chatbot, inputs=inp, outputs=out)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.36.1, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://a7c727725ab1968720.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
